# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Citation: {getattr(metadata, 'citeAs', '')}")


## 2. Data Overview
Review available record sets, fields, and their IDs ('@id').

In [ ]:
# List all available record sets by their @id
print("Available record sets (by @id):")
for record_set in metadata.record_sets:
    print(f"- {record_set['@id']} : {record_set.get('name', '')}")

# As an example, for exploration we will use the first record set:
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
if len(record_set_ids) == 0:
    raise ValueError("No record sets found in this dataset.")
main_record_set_id = record_set_ids[0]
main_record_set = next(rs for rs in metadata.record_sets if rs['@id'] == main_record_set_id)

# List all fields and columns for that record set by their @id
print(f"\nFields and columns for record set '{main_record_set_id}':")
for field in main_record_set.get('field', []):
    field_obj = field if isinstance(field, dict) else next(f for f in metadata.fields if f['@id'] == field)
    print(f"  Field: {field_obj['@id']}, name: {field_obj.get('name','')}")
    for column in field_obj.get('column', []):
        col_obj = column if isinstance(column, dict) else next(c for c in metadata.columns if c['@id'] == column)
        print(f"    Column: {col_obj['@id']} | in file: {col_obj.get('source', '')} | name: {col_obj.get('name','')}")


## 3. Data Extraction
Load data from the selected record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all available record sets
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} records.")
# For analysis, use the main (first) record set
print("\nColumns in main DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes using `@id` references for fields.

In [ ]:
# Choose a numeric field @id to analyze (example: coverage percentage, MW, etc.).
# We'll attempt to find a numeric field in the DataFrame columns (e.g. 'coverage', 'MW', 'pI', or peptide counts)
import numpy as np

# Helper to suggest numeric fields
sample_df = dataframes[main_record_set_id]
numeric_candidates = [col for col in sample_df.columns if sample_df[col].dtype in [np.float64, np.int64, np.float32, np.int32]]
if not numeric_candidates:
    # Try to coerce columns with likely numeric names
    likely_numeric = [col for col in sample_df.columns if any(part in col.lower() for part in ['mw', 'coverage', 'count', 'abundance', 'value', 'peptide', 'score', 'intensity', 'pI'])]
    for col in likely_numeric:
        try:
            sample_df[col] = pd.to_numeric(sample_df[col], errors='coerce')
        except Exception:
            continue
    numeric_candidates = [col for col in sample_df.columns if sample_df[col].dtype in [np.float64, np.int64, np.float32, np.int32]]

if not numeric_candidates:
    raise ValueError("No numeric-like fields found for EDA.")
numeric_field = numeric_candidates[0]
print(f"Using numeric field for analysis: {numeric_field}")

# Filter on this numeric field
threshold = sample_df[numeric_field].mean()  # Use the mean as an example threshold
filtered_df = sample_df[sample_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (mean): {len(filtered_df)} records")

# Normalize this numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nFirst few normalized records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely group field (e.g. description or accession or other categorical)
group_field_candidates = [col for col in sample_df.columns if sample_df[col].dtype == 'object' and col != numeric_field]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(f"\nGrouped data by '{group_field}':\n", grouped_df.head())
else:
    print("No suitable group field found for grouping analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(data=sample_df, x=numeric_field, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If grouping field exists, show top groups
if group_field:
    # Show top categories and their means
    top_n = 10
    top_groups = grouped_df.head(top_n)

    plt.figure(figsize=(10,6))
    sns.barplot(data=top_groups, x=numeric_field, y=group_field, palette="viridis")
    plt.title(f"Top {top_n} {group_field} by mean {numeric_field}")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.show()


## 6. Conclusion
This notebook introduced the [FAIR\(^2\)](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset loaded using the `mlcroissant` library, explored its metadata and structure via `@id`, selected a main record set, and performed example EDA and visualization steps using its numeric fields. You can extend the analysis by further exploring different fields or combining information across record sets using their `@id` references.